# Kedro + JupyterLab + MLflow: from data to predictions

This notebook is a client of the project. It uses the catalog, parameters, and code in `src/`; it does not contain a second implementation of training. Choose **Run → Run All Cells**. It creates the source when needed, runs experiments, and writes a new prediction batch.

The example uses synthetic housing prices in PLN: 1,500 labelled records and 250 records for prediction. The data is generated locally; no dataset is downloaded from the internet.

In [ ]:
from kedro_lasso.commands import prepare_runtime, seed
from kedro_lasso.settings import PROJECT_ROOT

prepare_runtime()
seed()
print(PROJECT_ROOT)

## 1. Load the Kedro project

`demo lab` starts JupyterLab through `kedro jupyter lab`. The kernel exposes `catalog`, `context`, `pipelines`, and `session`. The explicit reload below also lets the notebook run in a regular kernel from this environment.

Use `%reload_kedro` after changing code or YAML. One Kedro session handles one run; create a new session for each subsequent experiment.

In [ ]:
get_ipython().run_line_magic('load_ext', 'kedro.ipython')
get_ipython().run_line_magic('reload_kedro', f'\"{PROJECT_ROOT}\"')
print('Pipelines:', list(pipelines))
print('Datasets:', list(catalog.keys()))

## 2. Explore data through the catalog

The SQLite connection and SQL query are configured in `conf/base/catalog.yml` and `credentials.yml`. The notebook does not create a separate, parallel source connection.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

raw = catalog.load('raw_training')
display(raw.head())
display(raw.describe().T)
display(raw.isna().sum().rename('missing_values').to_frame())

In [ ]:
raw.plot.scatter(x='area_m2', y='price_pln', alpha=0.25, figsize=(8, 4))
plt.title('Synthetic housing prices')
plt.ylabel('Price [PLN]')
plt.show()

## 3. Reuse the same function as the Kedro node

Exploration uses domain code from the project. Validation checks columns, unique IDs, the target, and infinite values. Missing features are allowed: an imputer fitted only on training data will fill them.

In [ ]:
from kedro_lasso.domain import validate_frame

schema = context.params['data']
clean = validate_frame(raw, schema, training=True)
print('Shape:', clean.shape)
print('Features:', schema['features'])
print('Default alpha grid:', context.params['model']['alphas'])

## 4. Run the complete flow from the notebook

We override only the `alpha` grid and run name. The rest of the configuration stays in YAML. This is the same flow as `kedro run`: validation → 60/20/20 split → experiments → refit on train+validation → one held-out test evaluation → scoring → transaction in the output database.

Changing the grid creates new MLflow runs and a new model version. The test set does not participate in selecting `alpha`.

In [ ]:
from kedro.framework.session import KedroSession

with KedroSession.create(
    project_path=PROJECT_ROOT,
    runtime_params={
        'model': {'alphas': [0.1, 10.0, 100.0, 1000.0, 10000.0]},
        'tracking': {'run_name': 'notebook-sweep'},
    },
) as experiment_session:
    experiment_session.run()

# A new context sees the latest saved model version and report.
get_ipython().run_line_magic('reload_kedro', f'\"{PROJECT_ROOT}\"')
bundle = catalog.load('model_bundle')
report = bundle['report']
display(pd.DataFrame(report['trials']).sort_values('rmse'))
display(pd.DataFrame({
    'Lasso test': report['test_metrics'],
    'Baseline test': report['baseline_test_metrics'],
}))
print('Best alpha:', report['best_alpha'])
print('MLflow run_id:', report['run_id'])
print('Model URI:', report['model_uri'])

## 5. Compare MLflow experiments

In the UI at http://127.0.0.1:5000, open the `housing-lasso` experiment. The parent run contains test metrics, configuration, split IDs, and the model with its input signature. Child runs correspond to `alpha` values and contain validation metrics.

In [ ]:
import mlflow
from kedro_lasso.config import load_config

tracking = load_config()['parameters']['tracking']
mlflow.set_tracking_uri(tracking['uri'])
runs = mlflow.search_runs(experiment_names=[tracking['experiment_name']])
columns = [column for column in [
    'run_id', 'tags.mlflow.runName', 'tags.mlflow.parentRunId',
    'params.alpha', 'params.best_alpha', 'metrics.validation_rmse',
    'metrics.test_rmse', 'status',
] if column in runs.columns]
display(runs[columns].head(20))

In [ ]:
trials = pd.DataFrame(report['trials']).sort_values('alpha')
ax = trials.plot(x='alpha', y='rmse', marker='o', logx=True, legend=False, figsize=(8, 4))
ax.set(xlabel='alpha (log scale)', ylabel='Validation RMSE [PLN]',
       title='Regularisation selection on validation data')
plt.show()

coefficients = pd.Series(bundle['model']['lasso'].coef_, index=report['schema']['features'])
coefficients.sort_values().plot.barh(figsize=(8, 4))
plt.xlabel('Coefficient after standardisation [PLN]')
plt.title('Which features did Lasso retain?')
plt.show()

## 6. Results from the output database

`warehouse.db` has four tables: `model_runs`, `experiment_trials`, `prediction_batches`, and `predictions`. A batch has its own ID, input fingerprint, and relationship to the MLflow run. Read the results below through the Kedro catalog.

In [ ]:
predictions = catalog.load('warehouse_predictions')
display(predictions.head(10))
display(predictions.groupby('batch_id').agg(n_rows=('property_id', 'size')))
assert len(predictions) >= 250
assert predictions['prediction'].notna().all()

## 7. Score without retraining

`pipeline_name='scoring'` loads the latest model version. It uses the stored feature list and fitted preprocessing. It does not create new training experiments. A new prediction batch is created while prior results remain in the database.

In [ ]:
runs_before = len(mlflow.search_runs(experiment_names=[tracking['experiment_name']]))
with KedroSession.create(project_path=PROJECT_ROOT) as scoring_session:
    scoring_session.run(pipeline_name='scoring')
runs_after = len(mlflow.search_runs(experiment_names=[tracking['experiment_name']]))
assert runs_before == runs_after
get_ipython().run_line_magic('reload_kedro', f'\"{PROJECT_ROOT}\"')
display(catalog.load('warehouse_predictions').groupby('batch_id').size().rename('n_rows'))

## 8. Restore the model without Kedro

The MLflow artifact contains the imputer, scaler, and model. It supports independent inference in another process. Load only trusted artifacts: the local Kedro bundle and this MLflow model use Python serialization.

In [ ]:
import mlflow.sklearn
import numpy as np

restored_model = mlflow.sklearn.load_model(report['model_uri'])
scoring_data = catalog.load('raw_scoring')
features = report['schema']['features']
restored_predictions = restored_model.predict(scoring_data[features])
np.testing.assert_allclose(restored_predictions, bundle['model'].predict(scoring_data[features]))
display(pd.DataFrame({
    'property_id': scoring_data['property_id'],
    'prediction_pln': restored_predictions,
}).head())

## What to change next

- Features and hyperparameters: `conf/base/parameters.yml` or `runtime_params` in a new session.
- Fast training: `kedro run --env quick --pipeline training`.
- Another source: update the `source_db` connection and SQL in the catalog while preserving the column contract.
- Another output database: change the `warehouse_db` connection and install the relevant SQLAlchemy driver. The adapter has an explicit schema, keys, and transaction.
- Optional graph: `demo viz`, then http://127.0.0.1:4141.

See `docs/architecture.md` for project boundaries, database schema, and extension paths. After repeated experiments against a fixed test set, validate the final model on a new, independent dataset.